# NLU Shared Task — Demo Solution 1 (Category A)

This notebook generates predictions for the AV test set using the trained Solution 1 stacking ensemble.

**Pipeline:** Raw test CSV → spaCy preprocessing → stylometric features (84) → info-theoretic features (8) → General Impostor features (5) → stacking ensemble → predictions

In [ ]:
# Install required packages
!pip install spacy lightgbm textstat joblib tqdm scikit-learn pandas numpy scipy gdown
!python -m spacy download en_core_web_sm

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure src is in the Python path
sys.path.insert(0, os.path.abspath('..'))

## Setup: Download Required Model Artefacts
Before running inference, download the two large artefacts from Google Drive.
This only needs to run once.

In [ ]:
import gdown

models_dir = Path('../models/solution1')
features_dir = Path('../data/features')
models_dir.mkdir(parents=True, exist_ok=True)
features_dir.mkdir(parents=True, exist_ok=True)

# Download model_a2.joblib (LightGBM base model)
model_a2_path = models_dir / 'model_a2.joblib'
if not model_a2_path.exists():
    print('Downloading model_a2.joblib ...')
    gdown.download(
        url='https://drive.google.com/file/d/1rWwn__rn9BwFgdaxINDbklYOCgHquY1z/view?usp=sharing',
        output=str(model_a2_path),
        fuzzy=True,
    )
    print('Downloaded model_a2.joblib')
else:
    print('model_a2.joblib already present, skipping.')

# Download gi_corpus_vectors.joblib (GI impostor corpus)
corpus_path = features_dir / 'gi_corpus_vectors.joblib'
if not corpus_path.exists():
    print('Downloading gi_corpus_vectors.joblib ...')
    gdown.download(
        url='https://drive.google.com/file/d/193TCG1-I4zSeW2t0peWCi3hxAPZ2FIeX/view?usp=sharing',
        output=str(corpus_path),
        fuzzy=True,
    )
    print('Downloaded gi_corpus_vectors.joblib')
else:
    print('gi_corpus_vectors.joblib already present, skipping.')

## 1. Configure Paths

In [ ]:
TEST_CSV     = Path('../data/test.csv')
MODEL_DIR    = Path('../models/solution1')
FEATURES_DIR = Path('../data/features')
OUTPUT_PATH  = Path('../outputs/Group_17_A.csv')
N_JOBS       = 4

# Sanity checks
assert TEST_CSV.exists(),                                    f'Test CSV not found: {TEST_CSV}'
assert (MODEL_DIR / 'model_a2.joblib').exists(),             f'model_a2.joblib not found in {MODEL_DIR}'
assert (FEATURES_DIR / 'gi_corpus_vectors.joblib').exists(), f'gi_corpus_vectors.joblib not found in {FEATURES_DIR}'

print('All required files found.')
print(f'Test CSV: {pd.read_csv(TEST_CSV).shape[0]} pairs')

## 2. Generate Predictions

Runs the full pipeline end-to-end: preprocessing → feature extraction → stacking ensemble.

The GI features use the saved training corpus as the impostor pool (consistent with training).

In [ ]:
from src.solution1.predict import predict_from_csv

submission_df = predict_from_csv(
    test_csv=TEST_CSV,
    model_dir=MODEL_DIR,
    features_dir=FEATURES_DIR,
    n_jobs=N_JOBS,
)

print(f'Generated {len(submission_df)} predictions.')
print(f'Class distribution: {submission_df["prediction"].value_counts().to_dict()}')

## 3. Save Output

Saves predictions to a CSV with a single `prediction` column (0 or 1) as required by the submission spec.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission_df.to_csv(OUTPUT_PATH, index=False)

print(f'Saved to {OUTPUT_PATH}')
display(submission_df.head(10))